# Skip-gram untuk Klasifikasi Data Berita Menggunakan Naive Bayes

## 1. Import Library dan Membaca Dataset

Pada tahap ini dilakukan import library yang diperlukan untuk proses pengolahan teks, pembuatan word embedding menggunakan Word2Vec Skip-gram, klasifikasi menggunakan Naive Bayes, serta evaluasi model.

Dataset berita kemudian dibaca menggunakan pandas. Dataset terdiri dari 200 data dengan tiga kolom, yaitu id, isi_berita, dan label.

In [10]:
import pandas as pd
import numpy as np
import string
import re
import emoji

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from nltk.tokenize import word_tokenize

# Membaca dataset
df = pd.read_csv("dataset_berita.csv")
df

,id,isi_berita,label
0,1,Hamza Abdelkarim kagumi Erling Haaland. Strike...,sport
1,2,Kontrak Luke Shaw di Manchester United akan ha...,sport
2,3,Erling Haaland bermain bagus di Manchester Cit...,sport
3,4,Pengurus Pusat Kickboxing Indonesia (PP KBI) p...,sport
4,5,"Lina Hisage, atlet tolak peluru asal Wamena, P...",sport
...,...,...,...
195,196,"Direktur Utama MIND ID Maroef Sjamsoeddin, men...",finance
196,197,Purbaya Yudhi Sadewa menolak untuk menempati p...,finance
197,198,Purbaya Yudhi Sadewa telah melakukan proses se...,finance
198,199,PT Bayan Resources Tbk (BYAN) mengumumkan kead...,finance


## 2. Menyiapkan Stopword dan Stemming

Pada tahap ini digunakan library Sastrawi untuk melakukan preprocessing teks Bahasa Indonesia.

Stopword digunakan untuk menghapus kata-kata umum yang kurang memberikan informasi penting dalam proses klasifikasi. Sedangkan stemming digunakan untuk mengubah kata menjadi bentuk dasarnya.

Hasil dari tahap ini akan digunakan pada proses preprocessing berita.

In [11]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Membuat daftar stopword Bahasa Indonesia
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# Membuat stemmer Bahasa Indonesia
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

## 3. Preprocessing Teks

Tahap preprocessing dilakukan untuk membersihkan dan mempersiapkan teks sebelum digunakan dalam proses Word2Vec.

Tahapan yang dilakukan meliputi:
1. Mengubah seluruh teks menjadi huruf kecil.
2. Menghapus tanda baca.
3. Menghapus angka.
4. Melakukan tokenisasi dengan memisahkan teks menjadi kata-kata.
5. Menghapus stopword.
6. Melakukan stemming untuk mengubah kata menjadi bentuk dasarnya.

Hasil akhir dari proses ini disimpan dalam token_preprocessing dan akan digunakan sebagai data masukan untuk proses Skip-gram.

In [15]:
token_preprocessing = []

for teks in df["isi_berita"].dropna():

    # 1. Mengubah teks menjadi huruf kecil
    teks = teks.lower()

    # 2. Menghapus tanda baca
    teks = teks.translate(str.maketrans('', '', string.punctuation))

    # 3. Menghapus angka
    teks = re.sub(r'\d+', '', teks)

    # 4. Tokenisasi
    kata = teks.split()

    # 5. Menghapus stopword
    kata = [k for k in kata if k not in stopwords]

    # 6. Stemming
    kata = [stemmer.stem(k) for k in kata]

    token_preprocessing.append(kata)

# Menampilkan hasil preprocessing 3 berita pertama
for i, token in enumerate(token_preprocessing[:3], 1):
    print(f"Berita {i}:")
    print(token)
    print()

Berita 1:
['hamza', 'abdelkarim', 'kagum', 'erling', 'haaland', 'striker', 'muda', 'barcelona', 'mau', 'jadi', 'haaland', 'sungai', 'nil', 'hamza', 'abdelkarim', 'striker', 'muda', 'barcelona', 'asal', 'mesir', 'usia', 'tahun', 'musim', 'hamza', 'masuk', 'skuad', 'utama', 'barca', 'belum', 'main', 'tim', 'akademi', 'musim', 'lalu', 'baca', 'thierry', 'henry', 'suka', 'erling', 'haaland', 'hamza', 'abdelkarim', 'kagum', 'erling', 'haaland', 'bagi', 'haaland', 'striker', 'top', 'haaland', 'serang', 'hebat', 'luar', 'biasa', 'cetak', 'gol', 'selalu', 'ada', 'posisi', 'tepat', 'laku', 'buka', 'lansir', 'tribuna', 'selalu', 'ajar', 'main', 'hebat', 'amat', 'cara', 'main', 'aku', 'senang', 'juluk', 'haaland', 'sungai', 'nil', 'tegas', 'hamza', 'abdelkarim', 'mau', 'ajar', 'banyak', 'diri', 'mau', 'baik', 'beberapa', 'aspek', 'beri', 'percaya', 'lebih', 'latih', 'hansi', 'flick', 'aku', 'mau', 'cetak', 'banyak', 'gol', 'tang', 'trofi', 'halhal', 'perlu', 'aku', 'baik', 'aspekaspek', 'mana', '

## 4. Membuat Pasangan Target dan Konteks

Pada tahap ini dibuat pasangan kata target dan kata konteks sebagai bagian dari proses Skip-gram.

Parameter window = 1 digunakan sehingga setiap kata akan melihat satu kata di sebelah kiri dan satu kata di sebelah kanan sebagai konteks.

Hasilnya berupa tabel yang memiliki dua kolom, yaitu Target dan Konteks.

In [16]:
window = 1

data_target_context = []

for kalimat in token_preprocessing:

    for i, target in enumerate(kalimat):

        # Kata di sebelah kiri
        if i - window >= 0:
            data_target_context.append({
                "Target": target,
                "Konteks": kalimat[i - 1]
            })

        # Kata di sebelah kanan
        if i + window < len(kalimat):
            data_target_context.append({
                "Target": target,
                "Konteks": kalimat[i + 1]
            })

df_target_context = pd.DataFrame(data_target_context)

display(df_target_context)
print("Jumlah pasangan target-konteks:", len(df_target_context))

,Target,Konteks
0,hamza,abdelkarim
1,abdelkarim,hamza
2,abdelkarim,kagum
3,kagum,abdelkarim
4,kagum,erling
...,...,...
90169,penuh,masyarakat
90170,penuh,wajib
90171,wajib,penuh
90172,wajib,hnuega


Jumlah pasangan target-konteks: 90174


## 5. Membuat Model Word2Vec dengan Skip-gram

Pada tahap ini dibuat model Word2Vec menggunakan metode Skip-gram.

Parameter sg=1 menunjukkan bahwa Word2Vec menggunakan metode Skip-gram. Setiap kata akan direpresentasikan dalam bentuk vektor berdimensi 3 karena menggunakan vector_size=3.

Parameter window=2 digunakan untuk menentukan jumlah kata di sekitar kata target yang digunakan sebagai konteks. Kata yang muncul kurang dari 2 kali tidak dimasukkan ke dalam vocabulary karena menggunakan min_count=2.

Model dilatih selama 50 epoch.

In [19]:
model = Word2Vec(
    sentences=token_preprocessing,
    vector_size=3,
    window=2,
    min_count=2,
    sg=1,
    workers=1,
    epochs=50
)

print("Model Skip-gram berhasil dibuat")
print("Jumlah kata dalam vocabulary:", len(model.wv.index_to_key))

Model Skip-gram berhasil dibuat
Jumlah kata dalam vocabulary: 3315


## 6. Menampilkan Word Embedding

Setelah model Skip-gram selesai dilatih, setiap kata dalam vocabulary memiliki representasi berupa vektor.

Karena vector_size=3, setiap kata memiliki tiga nilai vektor, yaitu Vektor 1, Vektor 2, dan Vektor 3.

Hasil word embedding kemudian ditampilkan dalam bentuk DataFrame agar lebih mudah dilihat dan dianalisis.

In [21]:
data_vektor = []

for kata in model.wv.index_to_key:

    vektor = model.wv[kata]

    data_vektor.append(
        [kata] + vektor.tolist()
    )

df_vektor = pd.DataFrame(
    data_vektor,
    columns=[
        "Kata",
        "Vektor 1",
        "Vektor 2",
        "Vektor 3"
    ]
)

df_vektor

,Kata,Vektor 1,Vektor 2,Vektor 3
0,jadi,-0.777469,0.242221,0.891298
1,sebut,-0.555737,0.773251,0.988701
2,main,-1.384262,-0.598625,1.151997
3,jakarta,0.800397,0.558971,2.051014
4,indonesia,-0.756116,0.512971,0.851650
...,...,...,...,...
3310,amat,-0.960412,-0.234423,0.947376
3311,tribuna,-0.720344,-0.275172,1.158699
3312,akademi,-0.405817,-0.098585,1.049591
3313,nil,-1.078008,-0.326184,0.838722


## 7. Menyimpan Hasil Word Embedding

Hasil word embedding disimpan dalam format CSV dengan nama hasil_skipgram.csv.

File ini berisi kata-kata yang terdapat dalam vocabulary beserta nilai vektornya.

In [22]:
df_vektor.to_csv(
    "hasil_skipgram.csv",
    index=False
)

print("Hasil word embedding berhasil disimpan.")

Hasil word embedding berhasil disimpan.


## 8. Membuat Vektor Setiap Berita

Word2Vec menghasilkan vektor untuk setiap kata, sedangkan proses klasifikasi membutuhkan representasi untuk setiap berita.

Oleh karena itu, dibuat fungsi buat_vektor_berita() untuk mengubah kumpulan vektor kata dalam satu berita menjadi satu vektor berita.

Vektor berita diperoleh dengan menghitung nilai rata-rata dari seluruh vektor kata yang terdapat dalam berita.

In [23]:
def buat_vektor_berita(tokens, model):

    vektor_kata = []

    for kata in tokens:

        if kata in model.wv:
            vektor_kata.append(model.wv[kata])

    if len(vektor_kata) > 0:
        return np.mean(vektor_kata, axis=0)

    else:
        return np.zeros(model.vector_size)

## 9. Membentuk Vektor untuk Seluruh Berita

Pada tahap ini fungsi buat_vektor_berita() diterapkan pada seluruh berita dalam dataset.

Hasilnya disimpan dalam variabel X. Setiap baris mewakili satu berita, sedangkan setiap kolom mewakili dimensi dari vektor berita.

Dengan 200 berita dan vector_size=3, ukuran data yang dihasilkan adalah 200 × 3.

In [24]:
vektor_berita = []

for tokens in token_preprocessing:

    vektor = buat_vektor_berita(tokens, model)

    vektor_berita.append(vektor)

X = np.array(vektor_berita)

print("Ukuran data:", X.shape)

Ukuran data: (200, 3)


In [246]:
def buat_vektor_berita(tokens, model):

    vektor_kata = []

    for kata in tokens:
        if kata in model.wv:
            vektor_kata.append(model.wv[kata])

    if len(vektor_kata) > 0:
        return np.mean(vektor_kata, axis=0)

    else:
        return np.zeros(model.vector_size)

In [25]:
kolom_vektor = [
    f"Vektor {i}"
    for i in range(1, model.vector_size + 1)
]

df_vsm = pd.DataFrame(
    X,
    columns=kolom_vektor
)

df_vsm.insert(
    0,
    "id",
    df["id"].values
)

df_vsm["label"] = df["label"].values

display(df_vsm)

,id,Vektor 1,Vektor 2,Vektor 3,label
0,1,-1.177703,-0.217879,1.112942,sport
1,2,-1.445550,-0.308368,1.176569,sport
2,3,-1.488967,-0.377753,1.194824,sport
3,4,-0.771093,0.472988,1.130897,sport
4,5,-0.900863,0.436432,1.028400,sport
...,...,...,...,...,...
195,196,-0.192280,1.026721,1.448982,finance
196,197,-0.216425,-0.050297,1.555020,finance
197,198,0.112577,0.133109,1.767392,finance
198,199,-0.301265,1.090153,1.325101,finance


## 10. Menentukan Fitur dan Label

Pada tahap ini data dipisahkan menjadi fitur dan label.

Variabel X berisi tiga nilai vektor dari setiap berita yang digunakan sebagai fitur klasifikasi.

Variabel y berisi label berita yang menjadi target klasifikasi.

In [26]:
# Fitur berupa vektor berita
X = df_vsm[kolom_vektor].values

# Target berupa label berita
y = df_vsm["label"].values

print("Ukuran fitur:", X.shape)
print("Jumlah label:", len(y))

Ukuran fitur: (200, 3)
Jumlah label: 200


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data training :", X_train.shape)
print("Data testing  :", X_test.shape)

Data training : (160, 3)
Data testing  : (40, 3)


## 11. Klasifikasi Menggunakan Naive Bayes

Pada tahap ini digunakan algoritma Gaussian Naive Bayes untuk melakukan klasifikasi berita.

Model dilatih menggunakan data training yang terdiri dari vektor berita dan labelnya. Setelah proses pelatihan selesai, model digunakan untuk memprediksi label pada data testing.

Hasil prediksi disimpan dalam variabel y_pred.

In [28]:
nb = GaussianNB()

# Melatih model
nb.fit(X_train, y_train)

# Melakukan prediksi
y_pred = nb.predict(X_test)

print("Prediksi selesai.")

Prediksi selesai.


## 12. Evaluasi Menggunakan Accuracy dan Classification Report

Pada tahap ini dilakukan evaluasi terhadap hasil klasifikasi.

Akurasi digunakan untuk mengetahui persentase prediksi yang benar dari seluruh data testing.

Selain akurasi, digunakan classification report yang menampilkan nilai precision, recall, dan F1-score untuk setiap kelas.

Berdasarkan hasil pengujian, model memperoleh akurasi sebesar 95% pada 40 data testing.

In [30]:
akurasi = accuracy_score(y_test, y_pred)

print("Akurasi:", akurasi)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Akurasi: 0.95

Classification Report:
              precision    recall  f1-score   support

     finance       0.91      1.00      0.95        20
       sport       1.00      0.90      0.95        20

    accuracy                           0.95        40
   macro avg       0.95      0.95      0.95        40
weighted avg       0.95      0.95      0.95        40



## 13. Confusion Matrix

Confusion matrix digunakan untuk melihat jumlah prediksi benar dan salah pada masing-masing kelas.

Berdasarkan hasil confusion matrix, terdapat 20 data finance yang diprediksi dengan benar sebagai finance dan 18 data sport yang diprediksi dengan benar sebagai sport.

Terdapat 2 data sport yang diprediksi sebagai finance, sehingga jumlah prediksi yang benar adalah 38 dari 40 data testing.

In [31]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[20  0]
 [ 2 18]]


In [32]:
df_vsm.to_csv(
    "hasil_vector_space_model_skipgram.csv",
    index=False
)

print("Vector Space Model berhasil disimpan.")

Vector Space Model berhasil disimpan.
